# Preparation

## Import Modules 

In [1]:
from torchvision.datasets import ImageFolder
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

import torch.optim as optim
from tqdm.auto import tqdm
import cv2
import random

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(42)
random.seed(42)

ModuleNotFoundError: No module named 'torchvision'

## Define Dataset

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
#     transforms.RandomHorizontalFlip(p=0.5)
])

train_dataset = ImageFolder(root='./data/Kaggle/dataset/training_set/', transform=transform)
train_dataset = Subset(train_dataset, np.random.choice(range(8000), size=800, replace=False))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

test_dataset = ImageFolder(root='./data/Kaggle/dataset/test_set/', transform=transform)
test_dataset = Subset(test_dataset, np.random.choice(range(2000), size=200, replace=False))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True, num_workers=0)

NameError: name 'transforms' is not defined

## Load Model

In [ ]:
net = models.resnet50(pretrained=True)
# for param in net.parameters():
#     param.requires_grad = False
net.fc = nn.Linear(2048, 2)

## Set Environment

In [ ]:
device = torch.device('cuda')
net = net.to(device)

# Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=1e-4)

losses = {'train': [], 'val': []}
for epoch in tqdm(range(3)):
    train_losses = []
    for i, data in enumerate(tqdm(train_loader, leave=False)):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        losses['train'].append(loss.item())
    net.eval()
    with torch.no_grad():
        val_loss = np.mean([criterion(net(x.to(device)), y.to(device)).item() for x, y in test_loader])
    losses['val'].append(val_loss)
    net.train()
        
print('Finished Training')

## Plot Loss Graph

In [ ]:
epoch_x = np.arange(0, len(losses['train']), len(train_loader)) + len(train_loader)
plt.figure(figsize=(8, 5))
plt.plot(losses['train'])
plt.plot(epoch_x, losses['val'], 'o-')
plt.legend(losses.keys())
plt.xticks(epoch_x, np.arange(len(epoch_x))+1)
plt.ylabel('loss')
plt.xlabel('epoch')
plt.show()

## Evaluation

In [ ]:
from torchmetrics.functional import accuracy 

net.eval()
with torch.no_grad():
    train_preds = torch.cat([torch.stack([net(x.to(device)).argmax(dim=-1), y.to(device)], dim=1) for x, y in tqdm(train_loader, leave=False)], dim=0)
    train_accuracy = accuracy(train_preds[:,0], train_preds[:,1]).item()
    
    test_preds = torch.cat([torch.stack([net(x.to(device)).argmax(dim=-1), y.to(device)], dim=1) for x, y in tqdm(test_loader, leave=False)], dim=0)
    test_accuracy = accuracy(test_preds[:,0], test_preds[:,1]).item()
print(f'Train acc:{train_accuracy*100:.1f}%, Test acc: {test_accuracy*100:.1f}%')

# XAI
## Define Gradient-tracking Model

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Linear(2048, 2)
        
        self.features_conv = torch.nn.Sequential(*(list(self.resnet.children())[:-2]))
        
        self.avg_pool = torch.nn.Sequential(*(list(self.resnet.children())[-2:-1]))
        self.fc = torch.nn.Sequential(*(list(self.resnet.children())[-1:]))
        
        self.gradients = None
    
    def activations_hook(self, grad):
        self.gradients = grad
        
    def forward(self, x):
        x = self.features_conv(x)
        
        if self.train and x.requires_grad:
            h = x.register_hook(self.activations_hook)
        
        x = self.avg_pool(x)
        x = x.view(-1, 2048)
        x = self.fc(x)
        return x
    
    def get_activations_gradient(self):
        return self.gradients
    
    def get_activations(self, x):
        return self.features_conv(x)
    
net = Model()
net = net.to(device)

In [ ]:
def getGradCAM(X, pred, cls_):
    pred[:,cls_].backward()
    gradients = net.get_activations_gradient().cpu()
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    activations = net.get_activations(X).detach().cpu()
    for i in range(activations.shape[1]):
        activations[:, i, :, :] *= pooled_gradients[i]
    heatmap = torch.mean(activations, dim=1).squeeze()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= torch.max(heatmap)
    return cv2.resize(heatmap.squeeze().numpy(), (224, 224))

X = next(iter(test_loader))[0][:1].to(device)
pred = net(X)

img = np.transpose(X.cpu().numpy()[0], (1, 2, 0))
plt.imshow(img)
plt.imshow(getGradCAM(X, pred, pred.argmax().item()), alpha=0.5)
plt.show()